# 1.5) Pandas

Pandas is the workhorse for labelled, tabular data: the `Series` (a labelled 1D array) and the `DataFrame` (named columns sharing an index). This notebook loads a small table of daily station observations — air temperature and river discharge at two stations, with realistic sensor gaps — and works through reading, selecting, time-indexing, grouping, rolling windows, and joining. The running theme is that missing data is *information*, not a number to paper over, which is also where the generated-code bug at the end goes wrong.

:::{admonition} Learning objectives
:class: tip
- Build and inspect Series and DataFrames, and read a CSV robustly with typed, date-parsed columns.
- Select with .loc (labels), .iloc (positions), and boolean masks.
- Use a datetime index to resample to coarser periods and compute rolling windows.
- Aggregate with groupby, and combine tables with merge and concat.
- Treat missing values as physical information: detect with isna, and choose ffill or interpolate deliberately.
- Write results to disk with pathlib paths.
:::

## Series and DataFrame

A `Series` pairs values with an index; a `DataFrame` is a collection of columns sharing one index. Each column has its own dtype.

In [1]:
import numpy as np
import pandas as pd

# a Series: values with a labelled index
temps = pd.Series([18.2, 17.5, 19.1],
                  index=["2024-06-01", "2024-06-02", "2024-06-03"],
                  name="temp_celsius")
print(temps)

# a DataFrame: named columns sharing an index
df = pd.DataFrame({"temp_celsius": [18.2, 17.5, 19.1],
                   "discharge_m3s": [48.0, 51.2, 47.5]})
print(df)
print(df.dtypes.to_dict())

2024-06-01    18.2
2024-06-02    17.5
2024-06-03    19.1
Name: temp_celsius, dtype: float64
   temp_celsius  discharge_m3s
0          18.2           48.0
1          17.5           51.2
2          19.1           47.5
{'temp_celsius': dtype('float64'), 'discharge_m3s': dtype('float64')}


## Reading a CSV robustly

`read_csv` infers types, but for analysis you should be explicit: parse date columns to `datetime64`, and pin the dtype of key columns. First we generate an example file; csv *writing* is covered near the end.

In [2]:
# --- generate an example data file (uses tools covered later; just setup here) ---
rng = np.random.default_rng(0)
dates = pd.date_range("2024-06-01", periods=45, freq="D")
parts = []
for name, base_temp, base_q in [("BAS", 18.0, 50.0), ("LUG", 21.0, 30.0)]:
    parts.append(pd.DataFrame({
        "date": dates,
        "station": name,
        "temp_celsius": (base_temp + rng.normal(0, 1.5, 45)).round(1),
        "discharge_m3s": (base_q + rng.normal(0, 5, 45)).round(1),
    }))
raw = pd.concat(parts, ignore_index=True)
raw.loc[[3, 4, 50], "temp_celsius"] = np.nan       # simulate sensor gaps
raw.to_csv("station_observations.csv", index=False)
print("wrote station_observations.csv with", len(raw), "rows")

wrote station_observations.csv with 90 rows


In [3]:
obs = pd.read_csv(
    "station_observations.csv",
    parse_dates=["date"],            # -> datetime64, not object strings
    dtype={"station": "string"},     # pin the key column's type
)
print(obs.dtypes.to_dict())
print(obs.head())

{'date': dtype('<M8[us]'), 'station': <StringDtype(storage='python', na_value=<NA>)>, 'temp_celsius': dtype('float64'), 'discharge_m3s': dtype('float64')}
        date station  temp_celsius  discharge_m3s
0 2024-06-01     BAS          18.2           48.4
1 2024-06-02     BAS          17.8           57.3
2 2024-06-03     BAS          19.0           59.8
3 2024-06-04     BAS           NaN           59.0
4 2024-06-05     BAS           NaN           56.6


## Selecting: .loc, .iloc, and boolean masks

`.loc` selects by label, `.iloc` by integer position, and a boolean mask filters rows by condition.

In [4]:
# boolean indexing: warm days at Lugano
warm_lug = obs[(obs["station"] == "LUG") & (obs["temp_celsius"] > 22.0)]
print(warm_lug[["date", "temp_celsius"]].head())

# .iloc by position, .loc by label
print(obs.iloc[0].to_dict())          # first row, by position
print(obs.loc[0, "station"])          # one cell, by label

         date  temp_celsius
46 2024-06-02          23.4
56 2024-06-12          22.5
59 2024-06-15          22.3
71 2024-06-27          23.1
72 2024-06-28          22.1
{'date': Timestamp('2024-06-01 00:00:00'), 'station': 'BAS', 'temp_celsius': 18.2, 'discharge_m3s': 48.4}
BAS


## A datetime index: resample and rolling windows

Setting a `datetime` index unlocks time-aware operations. `resample` re-bins to a coarser period; `rolling` computes a moving window.

In [5]:
# index one station by date
bas = obs[obs["station"] == "BAS"].set_index("date").sort_index()

print("monthly mean °C:", bas["temp_celsius"].resample("MS").mean().round(2).tolist())
# rolling on the gap-free discharge: the first 6 are NaN while the window fills
print("7-day rolling mean discharge (m3 s-1):")
print(bas["discharge_m3s"].rolling(window=7).mean().round(2).head(10).tolist())

monthly mean °C: [17.82, 18.39]
7-day rolling mean discharge (m3 s-1):
[nan, nan, nan, nan, nan, nan, 53.84, 54.07, 53.5, 51.19]


## groupby: split, apply, combine

`groupby` splits rows by a key, applies an aggregation to each group, and combines the results.

In [6]:
summary = obs.groupby("station").agg(
    mean_temp=("temp_celsius", "mean"),     # mean skips NaN
    max_discharge=("discharge_m3s", "max"),
    n_obs=("temp_celsius", "size"),         # size counts every row, NaN included
)
print(summary.round(2))

         mean_temp  max_discharge  n_obs
station                                 
BAS          18.02           60.0     45
LUG          20.88           39.1     45


## Missing data as physical information

A gap is not a zero. pandas marks missing values as `NaN`, detects them with `isna`, and its reductions skip them by default. How you *fill* a gap is a modelling choice: `ffill` carries the last value forward (sensible for slowly varying state), `interpolate` draws a straight line between neighbours.

In [7]:
# missing values per column
print(obs.isna().sum().to_dict())

# two fills with different physical meaning
bas_temp = obs.loc[obs["station"] == "BAS", "temp_celsius"].reset_index(drop=True)
print("with gaps:  ", bas_temp.head(6).tolist())
print("ffill:      ", bas_temp.ffill().head(6).tolist())               # carry forward
print("interpolate:", bas_temp.interpolate().round(2).head(6).tolist())  # linear

{'date': 0, 'station': 0, 'temp_celsius': 3, 'discharge_m3s': 0}
with gaps:   [18.2, 17.8, 19.0, nan, nan, 18.5]
ffill:       [18.2, 17.8, 19.0, 19.0, 19.0, 18.5]
interpolate: [18.2, 17.8, 19.0, 18.83, 18.67, 18.5]


:::{admonition} Computational-thinking fundamental: missing is not zero
:class: important
A missing value means "we do not know", which is different from any measured number — and very different from zero, a real, often common, physical reading. Encode absence as `NaN` so that reductions skip it and gaps stay visible. Filling is a deliberate decision with consequences for every statistic computed afterwards; choose the method (carry-forward, interpolation, or leaving the gap) to match the physics, and never let a tool silently substitute zero.
:::

## Combining tables: merge and concat

`merge` joins tables on a shared key (a database-style join); `concat` stacks tables along an axis.

In [8]:
# join station metadata onto the observations
metadata = pd.DataFrame({
    "station": pd.array(["BAS", "LUG"], dtype="string"),
    "name": ["Basel-Binningen", "Lugano"],
    "elevation_m": [316, 273],
})
merged = obs.merge(metadata, on="station", how="left")
print(merged[["date", "station", "name", "elevation_m", "temp_celsius"]].head(3))

        date station             name  elevation_m  temp_celsius
0 2024-06-01     BAS  Basel-Binningen          316          18.2
1 2024-06-02     BAS  Basel-Binningen          316          17.8
2 2024-06-03     BAS  Basel-Binningen          316          19.0


:::{admonition} Quick exercise: a rolling discharge mean
:class: note
Index the Lugano rows by date, then compute the 7-day rolling mean of `discharge_m3s` and print its last value, rounded to one decimal.
:::

:::{admonition} Solution
:class: note dropdown
```python
lug = obs[obs["station"] == "LUG"].set_index("date").sort_index()
roll = lug["discharge_m3s"].rolling(window=7).mean()
print(round(roll.iloc[-1], 1))
```
:::

## Writing outputs

Save results to disk with a `pathlib` path, which keeps the code OS-independent.

In [9]:
from pathlib import Path

out_path = Path("station_summary.csv")
summary.to_csv(out_path)
print("wrote", out_path.name, "-", out_path.stat().st_size, "bytes")

wrote station_summary.csv - 99 bytes


## When generated code lies: filling gaps with zero

Asked to "clean and average" a column with gaps, an assistant fills the missing values with zero and then takes the mean. The code runs, but zero is a valid temperature, so the gaps become spurious cold readings that drag the mean down.

In [10]:
def mean_temperature(df):
    # fill missing values, then average (as an assistant returned it)
    clean = df["temp_celsius"].fillna(0)
    return clean.mean()

bas_obs = obs[obs["station"] == "BAS"]
print("buggy mean:  ", round(mean_temperature(bas_obs), 2))
print("correct mean:", round(bas_obs["temp_celsius"].mean(), 2))   # skips NaN
print("gaps filled with 0 °C:", int(bas_obs["temp_celsius"].isna().sum()))

buggy mean:   17.22
correct mean: 18.02
gaps filled with 0 °C: 2


:::{admonition} Diagnosis: zero is a real value, not "missing"
:class: warning
`fillna(0)` replaced unknown temperatures with 0 °C, a perfectly valid reading, so the average is biased toward zero — here by nearly a degree, with no error raised. pandas reductions already skip `NaN`, so the fill was not only wrong but unnecessary. If a gap-free series is genuinely required, `interpolate()` respects the surrounding values; zero almost never does.
:::

In [11]:
def mean_temperature(df):
    # reductions skip NaN already; interpolate only if a gap-free series is needed
    return df["temp_celsius"].mean()

print("fixed mean:", round(mean_temperature(bas_obs), 2))

fixed mean: 18.02


:::{admonition} Going deeper: time zones
:class: seealso dropdown
A naive timestamp has no zone; localize it, then convert.

```python
idx = pd.date_range("2024-06-01", periods=3, freq="h")
aware = idx.tz_localize("UTC")          # attach UTC
local = aware.tz_convert("Europe/Zurich")   # convert to local clock time
```

Store and compute in UTC; convert to local time only for display.
:::

:::{admonition} Going deeper: multi-index
:class: seealso dropdown
A hierarchical index lets one frame hold several grouping levels.

```python
multi = obs.set_index(["station", "date"]).sort_index()
print(multi.loc["BAS"].head())      # select an outer level
```

`groupby` often produces a multi-index automatically when you group by more than one key.
:::

:::{admonition} Going deeper: apply
:class: seealso dropdown
`apply` runs an arbitrary function per group or per row when no built-in aggregation fits.

```python
spread = obs.groupby("station")["temp_celsius"].apply(lambda s: s.max() - s.min())
```

Prefer vectorised built-ins (`mean`, `sum`, `agg`) where they exist; `apply` is slower and should be the fallback, not the default.
:::

:::{admonition} Going deeper: Polars as a fast alternative
:class: seealso dropdown
[polars](https://pola.rs/) is a newer dataframe library with a lazy, multi-threaded engine that is often much faster on large data, with a more explicit expression API.

```python
import polars as pl
df = pl.read_csv("station_observations.csv")
df.group_by("station").agg(pl.col("temp_celsius").mean())
```

The concepts transfer directly from pandas; the syntax differs. It is shown for reference and not run here.
:::

:::{admonition} Takeaways
:class: danger
- A Series is a labelled 1D array; a DataFrame is named columns sharing an index, each with its own dtype.
- Read CSVs explicitly: `parse_dates` for time columns, `dtype` for keys.
- Select with `.loc` (labels), `.iloc` (positions), and boolean masks; a datetime index enables `resample` and `rolling`.
- `groupby` is split-apply-combine; `merge` joins on a key, `concat` stacks.
- Missing is not zero: detect with `isna`, rely on NaN-skipping reductions, and fill with `ffill`/`interpolate` only as a deliberate physical choice.
- Filling gaps with `fillna(0)` silently biases statistics whenever zero is a valid value.
:::

## Resources

- [Python for Data Analysis, 3rd ed. — Getting Started with pandas](https://wesmckinney.com/book/pandas-basics) — McKinney; Series/DataFrame mechanics, selection, and the data-cleaning chapter that follows it.
- [pandas — Getting started](https://pandas.pydata.org/docs/getting_started/index.html) — the official task-oriented tutorials for reading, selecting, grouping, and combining data.
:::